<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Fine-Tune Computer Vision Models - Solution

---

### About
Fine-tuning is a method of modifying generalized CVs to have better performance in specific domains.

### Learning Objective
- Fine tune an image model

### Notebook Guide
- Setup the Model
   - System Setup
   - Setup your model
   - Create a custom dataset class
   - Setup data
- Fine Tune the model!
    - Training Setup
    - Training Loop
    - Monitoring the training metrics
    - Save the Model
    - Test the Model on New Images
- Try It!

## Installs

In [ ]:
# Install these as needed

# !pip install torch torchvision
# !pip install transformers
# !pip install matplotlib
# !pip install pycocotools

## Imports

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch import nn, optim
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, classification_report

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import random

In [ ]:
# Load an image from the data set to inspect it

healthy_image = Image.open("../data/grape-plants/Grape_healthy/0cd6cdbd-1674-4abb-b734-756da4994cd0___Mt.N.V_HL 6052_180deg.JPG")
healthy_image

In [ ]:
# Load an unhealthy image for comparison

rot_image= Image.open("../data/grape-plants/Grape_Black_rot/0a06c482-c94a-44d8-a895-be6fe17b8c06___FAM_B.Rot 5019_flipLR.JPG")
rot_image

## System Setup 
Check your system setup. GPU is required for real world fine tuning. This demo can be run on CPU. Note it takes about 10 minutes to run the training step on a standard personal computer with CPU.

In [ ]:
# Check you compute system
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Setup your model

Load a pre-trained model. Here we are using a `ViT` which is on the smaller side (will run faster) but is still effective. 

[Model Card](https://huggingface.co/google/vit-base-patch16-224-in21k)


Note: For CPU-only training, we're using a smaller model (`ViT-tiny`) to speed up training.

In [ ]:
# Use a smaller model for CPU training
model_name = "google/vit-base-patch16-224-in21k"
feature_extractor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification: healthy vs black rot
    ignore_mismatched_sizes=True
)

# Move model to the device (CPU in this case)
model = model.to(device)


## Create a custom dataset class

- You will process images on-the-fly rather than loading everything into memory at once, which is more memory efficient. 
  - On-the-fly processing is when the dataset class handles preprocessing each image at access time rather than preprocessing everything upfront, spreading the computational load throughout training. 
- It will also allow you to efficiently pull training and test data as well as data labels. 

In [ ]:
class GrapeLeafDataset(Dataset):
    def __init__(self, image_paths, labels, feature_extractor):
        self.image_paths = image_paths
        self.labels = labels
        self.feature_extractor = feature_extractor
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image and preprocess
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        pixel_values = self.feature_extractor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        
        # Get label
        label = self.labels[idx]
        
        return pixel_values, label

## Setup data 
1. Collect the labels from the image paths
2. Create training and validation datasets

In [ ]:
# Define classes and data paths
classes = ['Grape_healthy', 'Grape_Black_rot']
root_dir = "../data/grape-plants"
image_paths = []
labels = []


# Set seed for reproducibility
random.seed(42)

In [ ]:
# Collect image paths and labels
for class_idx, class_name in enumerate(classes):
    class_dir = os.path.join(root_dir, class_name)
    # Get all image files in the directory
    files = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'))]
    # Limit to 50 images per class for faster training
    selected_files = random.sample(files, min(50, len(files)))
    
    for img_name in selected_files:
        image_paths.append(os.path.join(class_dir, img_name))
        labels.append(class_idx)

In [ ]:
# Create train/validation split (80/20)
indices = list(range(len(image_paths)))
random.shuffle(indices)
train_size = int(0.8 * len(indices))

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_paths = [image_paths[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]
val_paths = [image_paths[i] for i in val_indices]
val_labels = [labels[i] for i in val_indices]

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")

In [ ]:
# Create data sets and dataloaders
train_dataset = GrapeLeafDataset(train_paths, train_labels, feature_extractor)
val_dataset = GrapeLeafDataset(val_paths, val_labels, feature_extractor)

# Smaller batch size for CPU training
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Fine Tune the model! 
## Training Setup

Define optimizer, loss function, and training loop

In our `train_epoch()` function below we will be fine tuning our base model (ViT here).  

- `loss.backward()`  # Calculates gradients
- `optimizer.step()` # Makes small adjustments to model's parameters based on the new data

The entire training loop (calling `train_epoch` repeatedly for multiple epochs) is where the fine-tuning magic happens. With each epoch, the model parameters are incrementally adjusted to better classify our specific grape disease images.

##### What's happening behind the scenes:

- The model is slowly adapting its understanding of image features from general objects to specific grape disease patterns.
- Early layers of the network (which detect basic features) change less.
- Later layers (which combine features into classifications) change more significantly.

In [ ]:
# Define training parameters
num_epochs = 5  # More epochs can improve results but will take longer
learning_rate = 2e-5  # Lower learning rate for fine-tuning

# Set up optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Define training and evaluation functions
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        # Move data to device
        images, labels = images.to(device), labels.to(device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(pixel_values=images)
        logits = outputs.logits
        
        # Calculate loss
        loss = criterion(logits, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Track stats
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        # Print progress every 5 batches
        if (batch_idx + 1) % 5 == 0:
            print(f"Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    # Calculate epoch metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():  # No gradients needed for evaluation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(pixel_values=images)
            logits = outputs.logits
            
            # Calculate loss
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Get predictions
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=classes)
    
    return avg_loss, accuracy, report

## Training Loop

In [ ]:
# Note: this can take almost 11 minutes to run - time for coffee!

# Track metrics

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# Start training
print(f"Starting training for {num_epochs} epochs...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)
    
    # Train for one epoch
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Training - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")
    
    # Evaluate on validation set
    val_loss, val_acc, report = evaluate(model, val_loader, criterion, device)
    print(f"Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")
    print("\nClassification Report:")
    print(report)
    
    # Save metrics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

## Monitoring the training metrics 
It is important to monitor the training and validation metrics - they show you how effectively the fine-tuning process is working as the model adapts from its general pre-training to your specific task.

_Be on the look out for the following:_

### Positive Patterns

- **Decreasing Loss**: Both training and validation loss should generally decrease over time. This indicates the model is learning useful patterns from the data.
- **Improving Accuracy**: Both training and validation accuracy should increase over time, showing that the model is getting better at classifying the images correctly.
- **Convergence**: Eventually, the metrics should start to plateau, indicating the model has learned what it can from the data.
- **Close Performance**: Ideally, training and validation metrics should be relatively close to each other, suggesting the model generalizes well.

### Warning Signs

- **Overfitting**: If training loss/accuracy continues to improve while validation metrics worsen or plateau, the model is likely memorizing the training data rather than learning generalizable patterns. This is visible as a widening gap between training and validation curves.
- **Underfitting**: If both training and validation metrics remain poor and show minimal improvement, the model may be too simple for the task, or the learning rate might be too low.
- **Erratic Validation Metrics**: If validation metrics jump around significantly between epochs, it could indicate that the validation set is too small or not representative.
- **Too-Perfect Training**: If training accuracy quickly reaches 100% while validation lags behind, this is a clear sign of overfitting.

### Note for the demo
Due to our small validation set (to support CPU processing) our accuracy is high (1.0) immediately. In the real world this would indicate that we may not have representative data or may have a data leakage issue. For this example we will ignore it. 

In [ ]:
# Plot loss and accuracy curves
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Training')
plt.plot(val_accuracies, label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## Save the model
The model will save in the `02-cv-fine-tuning` folder.

In [ ]:
# Save the model
model_save_path = "grape_disease_model"
model.save_pretrained(model_save_path)
feature_extractor.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

## Test the model on new images

In [ ]:
def predict_image(image_path, model, feature_extractor, device):
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    inputs = feature_extractor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device)
    
    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        prediction = torch.argmax(logits, dim=1).item()
    
    # Get class name and probability
    class_name = classes[prediction]
    probability = probabilities[0][prediction].item()
    
    return class_name, probability, image

In [ ]:
# Test on sample images
test_images = [
    # Add paths to test images here
    # For example:
    "../data/grape-plants/Grape_healthy/0cd6cdbd-1674-4abb-b734-756da4994cd0___Mt.N.V_HL 6052_180deg.JPG",
    "../data/grape-plants/Grape_Black_rot/0a06c482-c94a-44d8-a895-be6fe17b8c06___FAM_B.Rot 5019_flipLR.JPG"
]

In [ ]:
# plot predictions

plt.figure(figsize=(15, 10))

for i, img_path in enumerate(test_images):
    class_name, prob, img = predict_image(img_path, model, feature_extractor, device)
    
    plt.subplot(1, len(test_images), i+1)
    plt.imshow(img)
    plt.title(f"Prediction: {class_name}\nConfidence: {prob:.2f}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Try It! 
Select new images and test the model.

In [ ]:
# Test on sample images

test_images = [
    # Add paths to test images here
    # For example:
    "../data/grape-plants/Grape_healthy/1fdf78d5-6036-4c48-b395-d41868e89f46___Mt.N.V_HL 9102_new30degFlipLR.JPG",
    "../data/grape-plants/Grape_Black_rot/8c8ddeba-8c03-47ea-8551-c2730890548f___FAM_B.Rot 3399_flipLR.JPG"
]

# plot predictions

plt.figure(figsize=(15, 10))

for i, img_path in enumerate(test_images):
    class_name, prob, img = predict_image(img_path, model, feature_extractor, device)
    
    plt.subplot(1, len(test_images), i+1)
    plt.imshow(img)
    plt.title(f"Prediction: {class_name}\nConfidence: {prob:.2f}")
    plt.axis('off')

plt.tight_layout()
plt.show()